# Module 03 — AI Agents
## Lesson 4 — State and Memory

Agent frameworks often use the words **state**, **history**, **context**, and **memory** almost interchangeably. In production systems they should be separated.

> **State is what the current run needs. Memory is what the application deliberately carries into a future run.**

This lesson builds both mechanisms explicitly before introducing any memory framework or vector database.


### Why state exists at all

In Lesson 3, each model decision depended on what happened in previous steps. The application therefore needed to preserve the protocol history: user goal, model tool requests, tool observations, and the latest model output.

That is **working state**. If you discard it after every tool call, the model cannot reason from what it has already observed.

A useful mental model is:

```text
Agent run
  ├── goal
  ├── step counter
  ├── model/tool protocol history
  ├── observations
  └── temporary working facts
```


### State is broader than chat history

A list of messages is only one possible part of state. Production agents often also need deterministic application metadata that should not be hidden inside prose: request IDs, user identity, permissions, step budgets, tool results, workflow status, timestamps, and references to external resources.

Keeping these as structured fields makes them easier to inspect, validate, test, persist, and secure.


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
model = os.getenv("OPENAI_MODEL", "gpt-5.6")


## Example 1 — Make working state explicit

We will represent the current agent run with a small dataclass instead of a loose collection of variables. The model-facing protocol history remains in `input_items`, while application-owned metadata stays in normal Python fields.


In [ ]:
@dataclass
class AgentState:
    goal: str
    max_steps: int = 5
    step: int = 0
    input_items: list[Any] = field(default_factory=list)
    observations: list[dict[str, Any]] = field(default_factory=list)

    def __post_init__(self) -> None:
        if not self.input_items:
            self.input_items.append({"role": "user", "content": self.goal})

state = AgentState(
    goal="Help me plan a comfortable day in Melbourne tomorrow.",
)
state


The important separation is that `step` and `max_steps` are **host control data**. They are not facts the model gets to redefine. `observations` gives us an application-friendly audit trail, while `input_items` preserves the provider protocol required for the next model call.


In [ ]:
def record_observation(
    state: AgentState,
    *,
    tool_name: str,
    arguments: dict[str, Any],
    result: dict[str, Any],
) -> None:
    state.observations.append(
        {
            "step": state.step,
            "tool": tool_name,
            "arguments": arguments,
            "result": result,
        }
    )

record_observation(
    state,
    tool_name="get_weather",
    arguments={"city": "Melbourne", "date": "tomorrow"},
    result={"rain_probability_percent": 70, "temperature_max_c": 17},
)
state.observations


This structured trace is easier to log or test than trying to recover facts later from natural-language messages. Frameworks often maintain a richer state object for exactly this reason.


## Memory is different

Suppose today's task finishes. Tomorrow the same user starts a new request. Should the new agent run receive every token from the old conversation? Usually no.

Some information may be worth retaining, such as a stable preference:

> I prefer vegetarian food and morning departures.

Other information is temporary and should disappear with the task:

> Melbourne has a 70% rain probability tomorrow.

The first may be useful **memory**. The second is a time-sensitive **observation** from the current state. Persisting both forever would be a design mistake.


## Example 2 — A tiny durable memory store

We will use JSON on disk so persistence is completely visible. This is not the recommended database architecture for a large application; it is a teaching implementation that exposes the contract a database-backed memory service would later provide.


In [ ]:
class JsonMemoryStore:
    def __init__(self, path: Path) -> None:
        self.path = path

    def _load_all(self) -> dict[str, dict[str, str]]:
        if not self.path.exists():
            return {}
        return json.loads(self.path.read_text(encoding="utf-8"))

    def get_preferences(self, user_id: str) -> dict[str, str]:
        return self._load_all().get(user_id, {})

    def set_preference(self, user_id: str, key: str, value: str) -> None:
        data = self._load_all()
        data.setdefault(user_id, {})[key] = value
        self.path.write_text(
            json.dumps(data, indent=2, sort_keys=True),
            encoding="utf-8",
        )

    def delete_user(self, user_id: str) -> None:
        data = self._load_all()
        data.pop(user_id, None)
        self.path.write_text(json.dumps(data, indent=2), encoding="utf-8")

memory = JsonMemoryStore(Path("lesson-04-memory.json"))
memory.set_preference("demo-user", "diet", "vegetarian")
memory.set_preference("demo-user", "departure_time", "morning")
memory.get_preferences("demo-user")


A real service would add authentication, access control, encryption, retention policies, auditability, concurrency handling, and a proper persistence layer. The useful abstraction, however, is already visible: the rest of the agent should not care whether memory lives in JSON, PostgreSQL, Redis, or another store.


### Selectively inject memory into a new run

Memory only affects the model if the application chooses to retrieve it and place it in the model's context. There is no magical hidden memory inside the model API.

For a travel request, dietary and departure-time preferences are relevant. For a code-review agent, those same preferences should probably not be injected at all.


In [ ]:
def build_memory_context(preferences: dict[str, str]) -> str:
    if not preferences:
        return "No saved user preferences."

    lines = [f"- {key}: {value}" for key, value in sorted(preferences.items())]
    return "Saved user preferences:\n" + "\n".join(lines)

preferences = memory.get_preferences("demo-user")
memory_context = build_memory_context(preferences)
print(memory_context)


### Use memory in a fresh model call

This is deliberately a **new** request rather than a continuation of the old protocol history. The previous agent state is gone; only selected durable memory is carried forward.


In [ ]:
new_goal = (
    "Suggest a simple plan for flying to Sydney for a one-day conference, "
    "including when I should prefer to depart and what kind of lunch to look for."
)

response = client.responses.create(
    model=model,
    instructions=(
        "You are a concise travel assistant. "
        "Use saved preferences when they are relevant, but do not invent preferences.\n\n"
        + memory_context
    ),
    input=[{"role": "user", "content": new_goal}],
)

print(response.output_text)


The answer should prefer a morning departure and vegetarian lunch options because those preferences were explicitly retrieved from durable memory. Notice that no old weather observation or old protocol history was carried into the new request.


## Four useful memory categories

You will encounter several forms of memory in agent systems:

1. **Working state / short-term context** — what the current run needs right now.
2. **Episodic memory** — selected records of previous interactions or events.
3. **Semantic memory** — durable facts or preferences, such as a user's preferred language.
4. **Procedural memory** — durable instructions or learned operating rules about how a task should be performed.

These labels are useful, but production architecture should still ask a simpler question first: **what data is being persisted, why, for how long, and who can retrieve it?**


## Why not just keep the entire conversation forever?

Because unlimited history creates several problems: context-window pressure, higher token cost, slower requests, stale facts, contradictory information, privacy exposure, prompt-injection persistence, and poor relevance.

Memory should normally be **selected, scoped, revisable, and deletable** rather than an append-only dump of everything a user ever said.


## Context window is not memory storage

A model can only reason over information included in its current request or otherwise made available through provider-specific mechanisms. A long context window lets you send more information, but it does not solve persistence, relevance, correction, deletion, ownership, or retrieval.

A useful architecture is:

```text
Durable store
     ↓ retrieve relevant memory
Context builder
     + current goal
     + current working state
     ↓
Model call
```


## Production checklist

Before adding long-term memory, decide: what qualifies for persistence; whether memory is user-, tenant-, project-, or task-scoped; how stale facts are updated; how users can inspect or delete stored data; how secrets and sensitive information are excluded; how retrieval is authorised; how memory is protected from prompt injection; and how retention affects compliance and cost.

> **More memory is not automatically a better agent. Better memory means carrying forward the minimum useful information with explicit ownership and lifecycle rules.**


## Exercise

1. Add a `seat_preference` memory for the demo user and run the fresh travel request again.
2. Add an obviously irrelevant preference such as `editor_theme=dark`. Change `build_memory_context` so travel requests do not receive it.
3. Call `delete_user("demo-user")` and verify that a new session no longer receives those preferences.
4. Think about a production system you know: list which fields belong in current state and which, if any, deserve durable memory.

The next lesson moves from remembering information to **planning**: how an agent can decompose a goal, whether plans should be explicit, and when planning helps or hurts.
